# BOBs preprocessing — SSL3D / nnU-Net

**Dataset:** BOBs `resampled75` · **Dataset ID:** 102 · **Modality:** T2w · **Labels:** BOBs9

Use the existing participant-level splits: **31 / 10 / 10 participants** (42 train, 15 validation, 14 test scans). Run the cells in order from the project environment on Windows; Docker Desktop is required for the nnU-Net/SSL3D stages.

This notebook calls the reusable Python scripts in `scripts/`. The original data are not modified.

In [1]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BOBS_DIR = ROOT / "data" / "BOBs"
SPLITS = ROOT / "data" / "splits" / "bobs_splits.json"
OUTPUT = ROOT / "data" / "nnUNet_raw" / "Dataset102_BOBs"
SCRIPT = ROOT / "scripts" / "convert_bobs_to_nnunet.py"

print("BOBs source:", BOBS_DIR)
print("Splits:", SPLITS)
print("Output:", OUTPUT)
assert SPLITS.is_file()
assert SCRIPT.is_file()

BOBs source: C:\Users\yasmi\-brain-mri-foundation-models\data\BOBs
Splits: C:\Users\yasmi\-brain-mri-foundation-models\data\splits\bobs_splits.json
Output: C:\Users\yasmi\-brain-mri-foundation-models\data\nnUNet_raw\Dataset102_BOBs


## Step 1 — Convert BOBs to nnU-Net raw format

### 1.1 Select the correct source and dry-run

Use **`data/BOBs/resampled75/`**, not `neonatal_bobsV1.0`. Check matching T2w/BOBs9 pairs and participant splits **without writing files**.

**Expected:** 42 train, 15 validation, 14 test scans; return code `0`.

In [8]:
BOBS_DIR = ROOT / "data" / "BOBs" / "resampled75"
print("BOBs directory:", BOBS_DIR)
print("Exists:", BOBS_DIR.exists())

BOBs directory: C:\Users\yasmi\-brain-mri-foundation-models\data\BOBs\resampled75
Exists: True


In [ ]:
result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--bobs-dir", str(BOBS_DIR),
        "--dry-run",
    ],
    cwd=ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

### 1.2 Run the conversion

Run only after the dry-run succeeds. The conversion copies the T2w images and remaps segmentation labels to `0–7` for nnU-Net. Original inputs remain unchanged.

In [11]:
result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--bobs-dir", str(BOBS_DIR),
    ],
    cwd=ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

assert result.returncode == 0, "BOBs conversion failed!"

Participants: {'train': 31, 'val': 10, 'test': 10}
Scans: {'train': 42, 'val': 15, 'test': 14}
CONVERSION COMPLETE: {'train': 42, 'val': 15, 'test': 14} -> C:\Users\yasmi\-brain-mri-foundation-models\data\nnUNet_raw\Dataset102_BOBs


Return code: 0


### 1.3 Verify converted file counts

Check the raw nnU-Net folders: **42 / 15 / 14** MRI–mask pairs across train, validation, and test.

In [12]:
OUTPUT_DIR = ROOT / "data" / "nnUNet_raw" / "Dataset102_BOBs"

expected = {
    "imagesTr": 42, "labelsTr": 42,
    "imagesVal": 15, "labelsVal": 15,
    "imagesTs": 14, "labelsTs": 14,
}

for folder, count in expected.items():
    actual = len(list((OUTPUT_DIR / folder).glob("*.nii.gz")))
    print(f"{folder}: {actual}/{count}")
    assert actual == count, f"Unexpected count in {folder}"

print("\nALL 71 BOBs SCANS CONVERTED SUCCESSFULLY!")

imagesTr: 42/42
labelsTr: 42/42
imagesVal: 15/15
labelsVal: 15/15
imagesTs: 14/14
labelsTs: 14/14

ALL 71 BOBs SCANS CONVERTED SUCCESSFULLY!


## Step 2 — Fingerprint and experiment planning

Use the official nnU-Net Docker environment to compute the fingerprint and plans from the **42 training scans only**. Validation and test stay separate. `--no_pp` means **planning only**, not preprocessing.

In [13]:

from pathlib import Path
import subprocess

# Project root (already defined in the notebook)
RAW_DIR = ROOT / "data" / "nnUNet_raw"
PREPROCESSED_DIR = ROOT / "data" / "nnUNet_preprocessed"
RESULTS_DIR = ROOT / "data" / "nnUNet_results"

# Create output directories
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify converted BOBs dataset exists
assert (RAW_DIR / "Dataset102_BOBs" / "dataset.json").exists(), (
    "Dataset102_BOBs not found. Run Step 1 first."
)

command = [
    "docker", "run", "--rm",
    "-v", f"{RAW_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_raw",
    "-v", f"{PREPROCESSED_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_preprocessed",
    "-v", f"{RESULTS_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_results",
    "petermcgor/nnunetv2:1.1.0-nnssl",
    "nnUNetv2_plan_and_preprocess",
    "-d", "102",
    "--no_pp",
    "-np", "2",
]

print("Starting BOBs fingerprint + planning...", flush=True)

result = subprocess.run(
    command,
    cwd=ROOT,
)

assert result.returncode == 0, "Fingerprint/planning failed!"

print("BOBs fingerprint + planning completed!")


Starting BOBs fingerprint + planning...
BOBs fingerprint + planning completed!


### 2.1 Verify fingerprint and plans

Confirm that `dataset_fingerprint.json` and `nnUNetPlans.json` exist and display the planned configurations.

In [14]:

import json

PLANS_DIR = (
    ROOT / "data" / "nnUNet_preprocessed" / "Dataset102_BOBs"
)

fingerprint_path = PLANS_DIR / "dataset_fingerprint.json"
plans_path = PLANS_DIR / "nnUNetPlans.json"

assert fingerprint_path.exists(), "Missing fingerprint!"
assert plans_path.exists(), "Missing nnU-Net plans!"

with open(plans_path, encoding="utf-8") as f:
    plans = json.load(f)

print("Fingerprint: OK")
print("nnU-Net plans: OK")

for name, config in plans.get("configurations", {}).items():
    print(f"\nConfiguration: {name}")
    print("Spacing:", config.get("spacing"))
    print("Patch size:", config.get("patch_size"))
    print("Batch size:", config.get("batch_size"))
    print("Normalization:", config.get("normalization_schemes"))

print("\nSTEP 2 COMPLETED!")


Fingerprint: OK
nnU-Net plans: OK

Configuration: 2d
Spacing: [None, 0.75, 0.75]
Patch size: [256, 224]
Batch size: 61
Normalization: ['ZScoreNormalization']

Configuration: 3d_fullres
Spacing: [0.75, 0.75, 0.75]
Patch size: [128, 128, 128]
Batch size: 2
Normalization: ['ZScoreNormalization']

STEP 2 COMPLETED!


## Step 3 — SSL3D preprocessing: training set

Adapt Dataset102_BOBs to the pretrained SSL3D ResEncL checkpoint with `like_pretrained` and the existing CPU checkpoint-loading fix.

**Expected output:** 1 mm isotropic spacing, Z-score normalization, **42 MRI `.b2nd` + 42 mask `.b2nd` + 42 `.pkl`** files.

In [15]:

from pathlib import Path
import subprocess

IMAGE = "petermcgor/nnunetv2:1.1.0-nnssl"

RAW_DIR = ROOT / "data" / "nnUNet_raw"
PREPROCESSED_DIR = ROOT / "data" / "nnUNet_preprocessed"
RESULTS_DIR = ROOT / "data" / "nnUNet_results"

CPU_FIX = ROOT / "scripts" / "docker_fixes" / "like_nnssl.py"

assert CPU_FIX.is_file(), "Missing CPU checkpoint loading fix"
assert (PREPROCESSED_DIR / "Dataset102_BOBs" / "nnUNetPlans.json").exists()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = (
    "/opt/nnunet_resources/models/"
    "BaseMAETrainerExtendedHealth__nnsslPlans__onemmiso/"
    "fold_all/checkpoint_best.pth"
)

command = [
    "docker", "run", "--rm",
    "-v", f"{RAW_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_raw",
    "-v", f"{PREPROCESSED_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_preprocessed",
    "-v", f"{RESULTS_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_results",
    "-v", (
        f"{CPU_FIX.resolve().as_posix()}:"
        "/opt/nnunet_adaptation/nnunetv2/"
        "experiment_planning/like_nnssl.py:ro"
    ),
    IMAGE,
    "nnUNetv2_preprocess_like_nnssl",
    "-d", "102",
    "-n", "SSL3D_BOBs",
    "-pc", checkpoint,
    "-am", "like_pretrained",
    "-np", "1",
]

print("Starting SSL3D preprocessing for BOBs TRAIN...", flush=True)

subprocess.run(command, cwd=ROOT, check=True)

print("SSL3D BOBs training preprocessing completed!")


Starting SSL3D preprocessing for BOBs TRAIN...
SSL3D BOBs training preprocessing completed!


### 3.1 Verify training outputs

Check the SSL3D plans and confirm all **42 training cases** have an image, segmentation, and metadata file.

In [16]:

import json

DATASET_DIR = (
    ROOT / "data" / "nnUNet_preprocessed" / "Dataset102_BOBs"
)

train_dir = (
    DATASET_DIR / "Spacing__1.00_1.00_1.00___Norm__Z"
)

plans_files = list(DATASET_DIR.glob("ptPlans__SSL3D_BOBs*.json"))

print("SSL3D plans:", [p.name for p in plans_files])
print("Training folder exists:", train_dir.is_dir())

assert len(plans_files) == 1, "Expected one SSL3D BOBs plans file"
assert train_dir.is_dir(), "SSL3D training folder missing"

images = list(train_dir.glob("*.b2nd"))
masks = list(train_dir.glob("*_seg.b2nd"))
metadata = list(train_dir.glob("*.pkl"))

print("Total .b2nd files:", len(images))
print("Segmentation files:", len(masks))
print("Metadata files:", len(metadata))

assert len(images) == 84  # 42 MRI + 42 masks
assert len(masks) == 42
assert len(metadata) == 42

with open(plans_files[0], encoding="utf-8") as f:
    ssl3d_plans = json.load(f)

print("\nSSL3D TRAIN PREPROCESSING VERIFIED!")


SSL3D plans: ['ptPlans__SSL3D_BOBs____Spacing__1.00_1.00_1.00___Norm__Z.json']
Training folder exists: True
Total .b2nd files: 84
Segmentation files: 42
Metadata files: 42

SSL3D TRAIN PREPROCESSING VERIFIED!


## Step 4 — SSL3D preprocessing: validation and test

Apply the **same training-derived SSL3D plans** to the **15 validation** and **14 test** scans. Do not recompute the fingerprint or plans.

In [17]:

SCRIPT = ROOT / "scripts" / "preprocess_bobs_val_test.py"

command = [
    "docker", "run", "--rm",
    "-v", f"{RAW_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_raw",
    "-v", f"{PREPROCESSED_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_preprocessed",
    "-v", f"{SCRIPT.resolve().as_posix()}:/tmp/preprocess_bobs_val_test.py:ro",
    "petermcgor/nnunetv2:1.1.0-nnssl",
    "python", "/tmp/preprocess_bobs_val_test.py",
]

subprocess.run(command, cwd=ROOT, check=True)


CompletedProcess(args=['docker', 'run', '--rm', '-v', 'C:/Users/yasmi/-brain-mri-foundation-models/data/nnUNet_raw:/opt/nnunet_resources/nnUNet_raw', '-v', 'C:/Users/yasmi/-brain-mri-foundation-models/data/nnUNet_preprocessed:/opt/nnunet_resources/nnUNet_preprocessed', '-v', 'C:/Users/yasmi/-brain-mri-foundation-models/scripts/preprocess_bobs_val_test.py:/tmp/preprocess_bobs_val_test.py:ro', 'petermcgor/nnunetv2:1.1.0-nnssl', 'python', '/tmp/preprocess_bobs_val_test.py'], returncode=0)

## Step 5 — Automated preprocessing checks

Run `check_bobs_preprocessing.py` to validate the generated images, masks, labels, and metadata across all splits.

In [18]:

SCRIPT = ROOT / "scripts" / "check_bobs_preprocessing.py"

command = [
    "docker", "run", "--rm",
    "-v", f"{RAW_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_raw:ro",
    "-v", f"{PREPROCESSED_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_preprocessed:ro",
    "-v", f"{SCRIPT.resolve().as_posix()}:/tmp/check_bobs_preprocessing.py:ro",
    "petermcgor/nnunetv2:1.1.0-nnssl",
    "python", "/tmp/check_bobs_preprocessing.py",
]

subprocess.run(command, cwd=ROOT, check=True)


CompletedProcess(args=['docker', 'run', '--rm', '-v', 'C:/Users/yasmi/-brain-mri-foundation-models/data/nnUNet_raw:/opt/nnunet_resources/nnUNet_raw:ro', '-v', 'C:/Users/yasmi/-brain-mri-foundation-models/data/nnUNet_preprocessed:/opt/nnunet_resources/nnUNet_preprocessed:ro', '-v', 'C:/Users/yasmi/-brain-mri-foundation-models/scripts/check_bobs_preprocessing.py:/tmp/check_bobs_preprocessing.py:ro', 'petermcgor/nnunetv2:1.1.0-nnssl', 'python', '/tmp/check_bobs_preprocessing.py'], returncode=0)

## Step 6 — Visual quality control

Generate axial, coronal, and sagittal views of preprocessed T2w MRI volumes, with segmentation overlays. Inspect a few cases from each split for alignment and anatomy.

In [19]:

SCRIPT = ROOT / "scripts" / "visualize_bobs_preprocessing.py"
OUTPUT_QA = ROOT / "data" / "qa_bobs_ssl3d"

OUTPUT_QA.mkdir(parents=True, exist_ok=True)

command = [
    "docker", "run", "--rm",
    "-v", f"{PREPROCESSED_DIR.resolve().as_posix()}:/opt/nnunet_resources/nnUNet_preprocessed:ro",
    "-v", f"{OUTPUT_QA.resolve().as_posix()}:/opt/qa_output",
    "-v", f"{SCRIPT.resolve().as_posix()}:/tmp/visualize_bobs_preprocessing.py:ro",
    "petermcgor/nnunetv2:1.1.0-nnssl",
    "python", "/tmp/visualize_bobs_preprocessing.py",
    "--num-per-split", "3",
]

subprocess.run(command, cwd=ROOT, check=True)

print("BOBs visualizations generated!")


BOBs visualizations generated!


### 6.1 Final file-count check

Confirm that all **71 cases** have an MRI `.b2nd`, segmentation `.b2nd`, and metadata `.pkl` file (42 train, 15 validation, 14 test).

In [20]:
from pathlib import Path

base = ROOT / "data" / "nnUNet_preprocessed" / "Dataset102_BOBs"

for split, folder, expected in [
    ("Train", "Spacing__1.00_1.00_1.00___Norm__Z", 42),
    ("Validation", "SSL3D_val", 15),
    ("Test", "SSL3D_test", 14),
]:
    directory = base / folder
    images = list(directory.glob("*.b2nd"))
    masks = list(directory.glob("*_seg.b2nd"))
    metadata = list(directory.glob("*.pkl"))

    print(f"{split}: {len(images) - len(masks)}/{expected} MRI, "
          f"{len(masks)}/{expected} masks, "
          f"{len(metadata)}/{expected} metadata")

    assert len(images) - len(masks) == expected
    assert len(masks) == expected
    assert len(metadata) == expected

print("\nALL 71 BOBs SUBJECT SCANS VERIFIED!")

Train: 42/42 MRI, 42/42 masks, 42/42 metadata
Validation: 15/15 MRI, 15/15 masks, 15/15 metadata
Test: 14/14 MRI, 14/14 masks, 14/14 metadata

ALL 71 BOBs SUBJECT SCANS VERIFIED!
